Load Dataset

In [ ]:
import json
import os

coco_annotations_path = '/content/drive/MyDrive/FASDD_RS/annotations/COCO_RS_RGB'
images_path = '/content/drive/MyDrive/FASDD_RS/images'

# Find the COCO annotation JSON file
coco_json_file = None
for f in os.listdir(coco_annotations_path):
    if f.endswith('.json'):
        coco_json_file = os.path.join(coco_annotations_path, f)
        break

if coco_json_file:
    with open(coco_json_file, 'r') as f:
        coco_data = json.load(f)

    # Create a dictionary to map image filenames to their annotations
    image_annotations = {}
    for annotation in coco_data['annotations']:
        image_id = annotation['image_id']
        # Find the corresponding image filename
        image_filename = None
        for image_info in coco_data['images']:
            if image_info['id'] == image_id:
                image_filename = image_info['file_name']
                break

        if image_filename:
            if image_filename not in image_annotations:
                image_annotations[image_filename] = []
            image_annotations[image_filename].append(annotation)

    print(f"Loaded annotations for {len(image_annotations)} images.")

    # Now you can iterate through the images and their annotations
    # Example: Access annotations for a specific image
    # example_image_file = 'your_image.tif' # Replace with an actual image file
    # if example_image_file in image_annotations:
    #     print(f"\nAnnotations for {example_image_file}:")
    #     for annotation in image_annotations[example_image_file]:
    #         print(annotation)
    # else:
    #     print(f"\nNo annotations found for {example_image_file}")

else:
    print("COCO annotation JSON file not found.")

Pre-processing

In [ ]:
import cv2
import numpy as np

def normalize_image(image):
    """
    Normalizes image pixel values to [0, 1].

    Args:
        image (numpy.ndarray): The input image (expected to be in 0-255 range).

    Returns:
        numpy.ndarray: The normalized image with values in [0, 1].
    """
    # Assuming image is in uint8 format (0-255)
    normalized_image = image.astype(np.float32) / 255.0
    return normalized_image

def transform_color_space(image, target_color_space='HSV'):
    """
    Transforms the color space of an image.

    Args:
        image (numpy.ndarray): The input image (expected to be in BGR format for OpenCV).
        target_color_space (str): The target color space ('HSV' or 'YCbCr').

    Returns:
        numpy.ndarray: The image in the target color space.
    """
    if target_color_space == 'HSV':
        # OpenCV uses BGR by default, convert from RGB if necessary
        if image.shape[-1] == 3 and image.dtype == np.uint8:
             image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        return cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    elif target_color_space == 'YCbCr':
         if image.shape[-1] == 3 and image.dtype == np.uint8:
             image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
         return cv2.cvtColor(image, cv2.COLOR_BGR2YCR_CB) # Note: OpenCV uses YCrCb, not YCbCr
    else:
        raise ValueError("Unsupported target_color_space. Choose 'HSV' or 'YCbCr'.")


def preprocess_image(image_path):
    """
    Applies preprocessing steps to an image, including normalization and color space transformation.

    Args:
        image_path (str): The path to the input image.

    Returns:
        numpy.ndarray: The preprocessed image data.
    """
    try:
        # Load the image using OpenCV
        image = cv2.imread(image_path)
        if image is None:
            print(f"Error: Could not load image from {image_path}")
            return None

        # Apply preprocessing steps
        normalized_image = normalize_image(image)
        hsv_image = transform_color_space(normalized_image, target_color_space='HSV')
        # You can add other preprocessing steps here, like radiometric calibration or atmospheric correction

        preprocessed_image = hsv_image # Or whichever is the final output of your pipeline

        return preprocessed_image

    except Exception as e:
        print(f"An error occurred during preprocessing {image_path}: {e}")
        return None

# Example usage (you will need to adapt this to process your dataset)
# image_file = '/content/drive/MyDrive/FASDD_RS/images/neitherFireNorSmoke_RS000000.tif' # Replace with an actual image file from your dataset
# preprocessed_image_data = preprocess_image(image_file)
# if preprocessed_image_data is not None:
#     print("Preprocessing complete.")
#     print(f"Shape of preprocessed image: {preprocessed_image_data.shape}")
#     print(f"Data type of preprocessed image: {preprocessed_image_data.dtype}")
#     # You can display or further process the preprocessed image here